# 00 — Análise Exploratória: Receita Realizada por Fonte Detalhada

**Fonte de dados:** `receita_realizada_a_partir_2019_sgo.parquet`  
**Chave de série:** `Fonte_Det_Cód_Harm`  
**Valor:** `Valor_ReceitaRealizada`  
**Cobertura:** Jan/2019 → Out/2025 (até 82 observações mensais)

## Seções
1. Carregamento e visão geral
2. Cobertura temporal
3. Análise dimensional (categorias, origens, espécies)
4. Análise da dimensão Fonte Detalhada
5. Distribuição de valores
6. Construção das séries temporais
7. Sazonalidade e tendência
8. Top fontes e concentração de receita
9. Qualificação para modelagem
10. Síntese

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from IPython.display import display

from utils.preprocessing import carregar_dados, construir_series, mapear_nomes, filtrar_series
from utils.acf_analysis import analisar_serie
from utils.preprocessing import preparar_serie, dividir_serie
from config import FIM_SERIE, MIN_SERIE, HORIZONTE, MESES_PREV

CORES = {
    'azul'  : '#1B3A5C', 'azul_m': '#2E6DA4', 'azul_c': '#D6E8F7',
    'verde' : '#27AE60', 'verm'  : '#C0392B', 'amar'  : '#F39C12',
    'cinza' : '#BDC3C7', 'roxo'  : '#8E44AD', 'laran' : '#E67E22',
}
plt.rcParams.update({
    'figure.dpi': 130, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
})
print(f'FIM_SERIE={FIM_SERIE}  HORIZONTE={HORIZONTE}m  MESES_PREV={MESES_PREV[0]}..{MESES_PREV[-1]}')

## 1 · Carregamento e visão geral

In [ ]:
df = carregar_dados()
print(f'Shape: {df.shape}')
print(f'Colunas: {list(df.columns)}')
print()
print('Tipos:')
print(df.dtypes.to_string())
display(df.head(3))

In [ ]:
v = df['Valor_ReceitaRealizada']
print('=== Valor_ReceitaRealizada ===')
print(f'  Total registros : {len(v):,}')
print(f'  Zeros           : {(v==0).sum():,}  ({(v==0).mean()*100:.1f}%)')
print(f'  Negativos       : {(v<0).sum():,}  ({(v<0).mean()*100:.1f}%)')
print(f'  Positivos       : {(v>0).sum():,}  ({(v>0).mean()*100:.1f}%)')
print(f'  Min: {v.min():>20,.2f}')
print(f'  Max: {v.max():>20,.2f}')
print(f'  Media: {v.mean():>18,.2f}')
print(f'  Total receita: R$ {v.sum()/1e9:.3f} Bi')

## 2 · Cobertura temporal

In [ ]:
mensal = df.groupby('Periodo')['Valor_ReceitaRealizada'].agg(['sum','count']).reset_index()
mensal['Periodo_str'] = mensal['Periodo'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

ax = axes[0]
ax.fill_between(mensal['Periodo_str'], mensal['sum']/1e9,
                color=CORES['azul_m'], alpha=0.7)
ax.set_title('Receita Realizada Total por Mês (R$ Bilhões)', fontsize=11, fontweight='bold')
ax.set_ylabel('R$ Bilhões')
ax.tick_params(axis='x', rotation=45, labelsize=7)

ax2 = axes[1]
ax2.bar(mensal['Periodo_str'], mensal['count'],
        color=CORES['azul_c'], edgecolor=CORES['azul_m'])
ax2.set_title('Numero de Registros por Mes', fontsize=11, fontweight='bold')
ax2.set_ylabel('Registros')
ax2.tick_params(axis='x', rotation=45, labelsize=7)

plt.tight_layout()
plt.show()

## 3 · Análise dimensional

In [ ]:
dims = [
    ('Categ_Econ_Receita_Nome',  'Categoria Economica'),
    ('Origem_Receita_Nome',      'Origem da Receita'),
    ('Especie_Receita_Nome' if 'Especie_Receita_Nome' in df.columns else 'Espécie_Receita_Nome', 'Especie da Receita'),
]
# Normaliza nome de coluna
dims = [(c if c in df.columns else c.replace('_','\_'), n) for c, n in dims]
dims_ok = [(c, n) for c, n in dims if c in df.columns]

fig, axes = plt.subplots(1, len(dims_ok), figsize=(6*len(dims_ok), 5))
if len(dims_ok) == 1: axes = [axes]

for ax, (col, titulo) in zip(axes, dims_ok):
    top = df.groupby(col)['Valor_ReceitaRealizada'].sum().nlargest(8)
    labels = [str(l)[:25] for l in top.index]
    ax.barh(labels, top.values/1e6, color=CORES['azul_m'], edgecolor='white')
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.set_xlabel('R$ Milhoes')
    ax.invert_yaxis()

plt.suptitle('Receita Total por Dimensao (Top 8)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4 · Análise da dimensão Fonte Detalhada

In [ ]:
print(f'Fonte_Det_Cod_Harm: {df["Fonte_Det_Cód_Harm"].nunique()} codigos unicos')
print(f'Fonte_Det_Nome_Harm: {df["Fonte_Det_Nome_Harm"].nunique()} nomes unicos')
print()

top_fontes = (
    df.groupby(['Fonte_Det_Cód_Harm','Fonte_Det_Nome_Harm'])['Valor_ReceitaRealizada']
    .sum().reset_index()
    .sort_values('Valor_ReceitaRealizada', ascending=False)
    .head(15)
)
print('Top 15 Fontes Detalhadas por receita total:')
for _, row in top_fontes.iterrows():
    v = row['Valor_ReceitaRealizada']
    label = ('R$ {:.2f} Bi'.format(v/1e9) if abs(v)>=1e9 else
             'R$ {:.2f} Mi'.format(v/1e6) if abs(v)>=1e6 else
             'R$ {:,.0f}'.format(v))
    print(f"  {row['Fonte_Det_Cód_Harm']:12} {str(row['Fonte_Det_Nome_Harm'])[:55]:55s} {label}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
top15 = top_fontes.head(15)
labels = [str(r['Fonte_Det_Cód_Harm']) + ' — ' + str(r['Fonte_Det_Nome_Harm'])[:35]
          for _, r in top15.iterrows()]
cores_bar = plt.cm.Blues_r(np.linspace(0.3, 0.8, len(top15)))
ax.barh(labels, top15['Valor_ReceitaRealizada'].values/1e9, color=cores_bar)
ax.set_xlabel('Receita Total (R$ Bilhoes)')
ax.set_title('Top 15 Fontes Detalhadas — Receita Total Acumulada (2019-2025)', fontsize=11, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5 · Distribuição de valores

In [ ]:
v_pos = df.loc[df['Valor_ReceitaRealizada'] > 0, 'Valor_ReceitaRealizada']
v_neg = df.loc[df['Valor_ReceitaRealizada'] < 0, 'Valor_ReceitaRealizada'].abs()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(np.log10(v_pos + 1), bins=60, color=CORES['azul_m'], edgecolor='white', alpha=0.8)
axes[0].set_title('Distribuicao de Valores Positivos (log10)', fontsize=10, fontweight='bold')
axes[0].set_xlabel('log10(Valor + 1)')
axes[0].set_ylabel('Frequencia')

axes[1].hist(np.log10(v_neg + 1), bins=30, color=CORES['verm'], edgecolor='white', alpha=0.8)
axes[1].set_title('Distribuicao de Valores Negativos em modulo (log10)', fontsize=10, fontweight='bold')
axes[1].set_xlabel('log10(|Valor| + 1)')
axes[1].set_ylabel('Frequencia')

plt.suptitle(f'Positivos: {len(v_pos):,}  |  Negativos: {len(v_neg):,}  |  Zeros: {(df["Valor_ReceitaRealizada"]==0).sum():,}',
             fontsize=10)
plt.tight_layout()
plt.show()

## 6 · Construção das séries temporais

In [ ]:
series_raw = construir_series(df)
nomes      = mapear_nomes(df)
series, exc = filtrar_series(series_raw)

print(f'Total de codigos unicos : {len(series_raw)}')
print(f'Series validas          : {len(series)}  (fim={FIM_SERIE}, min_obs={MIN_SERIE})')
print(f'Series excluidas        : {len(exc)}')
print()

# Distribuicao do comprimento das series validas
lens = [len(s) for s in series.values()]
print(f'Comprimento - min={min(lens)}  median={int(np.median(lens))}  max={max(lens)}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lens, bins=20, color=CORES['azul_m'], edgecolor='white', rwidth=0.85)
ax.axvline(np.median(lens), color=CORES['verm'], lw=2, linestyle='--',
           label=f'Mediana = {int(np.median(lens))}')
ax.set_xlabel('Numero de observacoes')
ax.set_ylabel('Numero de series')
ax.set_title(f'Distribuicao do comprimento das {len(series)} series validas', fontsize=11, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 7 · Sazonalidade e tendência — Top 5 fontes

In [ ]:
# Top 5 por volume total entre as series validas
totais = {cod: s.sum() for cod, s in series.items()}
top5   = sorted(totais, key=totais.get, reverse=True)[:5]

fig, axes = plt.subplots(len(top5), 1, figsize=(14, 3*len(top5)))

for ax, cod in zip(axes, top5):
    s    = series[cod]
    nome = str(nomes.get(cod, cod))[:60]
    ax.fill_between(s.index.astype(str), s.values/1e6, alpha=0.5, color=CORES['azul_m'])
    ax.plot(s.index.astype(str), s.values/1e6, color=CORES['azul'], lw=1.5)
    ax.set_title(f'{cod} — {nome}', fontsize=9, fontweight='bold')
    ax.set_ylabel('R$ Milhoes')
    ax.tick_params(axis='x', rotation=45, labelsize=7)

plt.suptitle('Evolucao mensal — Top 5 Fontes Detalhadas por receita total', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8 · Curva de concentração (Pareto)

In [ ]:
totais_sorted = sorted(totais.values(), reverse=True)
total_geral   = sum(max(v, 0) for v in totais_sorted)
cum_pct = np.cumsum([max(v,0) for v in totais_sorted]) / total_geral * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cum_pct)+1), cum_pct, color=CORES['azul'], lw=2.5)
ax.fill_between(range(1, len(cum_pct)+1), cum_pct, alpha=0.15, color=CORES['azul_m'])

for pct_target in [50, 80]:
    idx = np.searchsorted(cum_pct, pct_target)
    ax.axhline(pct_target, color=CORES['verm'], lw=1, linestyle='--', alpha=0.7)
    ax.axvline(idx, color=CORES['verm'], lw=1, linestyle='--', alpha=0.7)
    ax.text(idx + 2, pct_target - 4, f'{idx} fontes = {pct_target}% receita', fontsize=9, color=CORES['verm'])

ax.set_xlabel('Numero de fontes detalhadas (ordenadas por volume decrescente)')
ax.set_ylabel('% cumulativa da receita total')
ax.set_title('Curva de Concentracao (Pareto) — Series Validas', fontsize=11, fontweight='bold')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## 9 · Qualificação para modelagem

In [ ]:
registros = []
for cod, serie in list(series.items()):
    nome = str(nomes.get(cod, '---'))
    serie_pronta, grupo_sinal, shift_value = preparar_serie(serie)
    treino_vals, teste_vals, treino_idx, teste_idx, n_holdout, n_lags = dividir_serie(serie_pronta)
    acf_info = analisar_serie(treino_vals)
    registros.append({
        'Fonte_Cód'    : cod,
        'Nome'         : nome[:60],
        'Grupo_Sinal'  : grupo_sinal,
        'N_Total'      : len(serie),
        'N_Treino'     : len(treino_vals),
        'N_Holdout'    : n_holdout,
        'N_Lags'       : acf_info['n_lags'] if not acf_info['e_ruido_branco'] else 0,
        'LB_Pvalor'    : round(acf_info['lb_pvalor'], 4),
        'Ruido_Branco' : acf_info['e_ruido_branco'],
        'Lags_Sig_PACF': str(acf_info['lags_sig_pacf']),
    })

qual_df = pd.DataFrame(registros)
n_rb  = qual_df['Ruido_Branco'].sum()
n_mod = (~qual_df['Ruido_Branco']).sum()

print(f'Series qualificadas para modelagem: {n_mod} / {len(qual_df)}')
print(f'Ruido branco (excluidas)          : {n_rb} / {len(qual_df)}')
print()
print('Distribuicao por grupo de sinal:')
print(qual_df['Grupo_Sinal'].value_counts().to_string())
display(qual_df.sort_values('Ruido_Branco').head(20))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Ruido branco vs modelavel
ax = axes[0]
counts = [n_rb, n_mod]
labels = [f'Ruido Branco\n({n_rb})', f'Modelavel\n({n_mod})']
ax.pie(counts, labels=labels, colors=[CORES['verm'], CORES['verde']],
       autopct='%1.1f%%', startangle=90)
ax.set_title('Qualificacao para Modelagem', fontsize=10, fontweight='bold')

# Distribuicao por grupo de sinal
ax2 = axes[1]
gs = qual_df[~qual_df['Ruido_Branco']]['Grupo_Sinal'].value_counts()
cores_gs = [CORES['azul_m'], CORES['verm'], CORES['amar']]
ax2.bar(gs.index, gs.values, color=cores_gs[:len(gs)], edgecolor='white')
ax2.set_title('Grupo de Sinal — Series Modelaveis', fontsize=10, fontweight='bold')
ax2.tick_params(axis='x', rotation=15)

# Distribuicao de lags selecionados
ax3 = axes[2]
lags_mod = qual_df[~qual_df['Ruido_Branco']]['N_Lags']
ax3.hist(lags_mod, bins=range(0, lags_mod.max()+2), color=CORES['azul_m'],
         edgecolor='white', rwidth=0.85)
ax3.set_title('Distribuicao de N_Lags — Series Modelaveis', fontsize=10, fontweight='bold')
ax3.set_xlabel('N_Lags')
ax3.set_ylabel('Numero de series')

plt.tight_layout()
plt.show()

## 10 · Síntese

| Dimensão | Valor |
|---|---|
| **Registros totais** | 75.013 |
| **Cobertura** | Jan/2019 → Out/2025 (82 meses) |
| **Chave de série** | `Fonte_Det_Cód_Harm` |
| **Séries únicas no parquet** | 2.277 |
| **Séries válidas** | 367 (fim=Out/2025, ≥10 obs.) |
| **Séries modeláveis** | ~165 (não-ruído-branco) |
| **Horizonte de previsão** | 12 meses (Nov/2025 → Out/2026) |

**Características notáveis:**
- Presença de valores negativos (~2%) — estornos e ajustes contábeis
- Alta concentração: poucas fontes representam a maioria da receita (curva de Pareto íngreme)
- Séries longas para as fontes principais (82 obs.) — bom para modelagem com sazonalidade
- ~55% das séries qualificadas para modelagem (resto é ruído branco — sem autocorrelação detectável)